# 05 · Fase 1 para las 10 categorías del núcleo: efecto propio y efectos dentro de la categoría

Estima el modelo de la Fase 1 de `metodologia.tex` para las **10 categorías del núcleo** de la tienda 44 (D‑28, D‑30): las seis **sin programa semanal** (BEVERAGES, GROCERY I, CLEANING, DAIRY, BREAD/BAKERY, LIQUOR,WINE,BEER) y las cuatro **con programa** (MEATS, POULTRY, DELI los viernes; PRODUCE los miércoles). En estas últimas se excluyen los días de programa (A‑03), así que sus efectos se refieren solo a las **promociones fuera del programa**.

$$
\mathbb{E}[y_{it}] = \exp\Big(
  \alpha_i + \gamma_{\text{dow}(t)} + \tau_{m(t)} + \mathbf{h}_t'\boldsymbol{\theta}
  + \sum_{b\in\mathcal{B}} \beta_b\, D^{(b)}_{it}
  + \sum_{b\in\mathcal{B}} \psi^{\text{cl}}_b\, S^{\text{cl}(b)}_{it}
  + \sum_{b\in\mathcal{B}} \psi^{\text{cat}}_b\, S^{\text{cat}(b)}_{it}
\Big), \qquad \mathcal{B}=\{0,\ \ell_{1\text{-}3},\ \ell_{4\text{-}6}\}
$$

| Símbolo | Columna | Qué es |
|---|---|---|
| $y_{it}$ | `unit_sales_ppml` | ventas del SKU $i$ el día $t$ (con ceros; devoluciones llevadas a 0) |
| $\alpha_i$, $\gamma_{\text{dow}}$, $\tau_m$ | `item_nbr`, `dow`, `mes` | efectos fijos de SKU, día de la semana y año‑mes (D‑16) |
| $\mathbf{h}_t$ | `dia_libre`, `adicional`, … , `post_pago` | calendario de la tienda 44 (D‑10, D‑13) |
| $D_{it}$ | `onpromotion` y sus bloques `_l1_3`, `_l4_6` | promoción propia: el día y el promedio de los días 1–3 y 4–6 anteriores (D‑14) |
| $S^{\text{cl}}_{it}$ | `promo_clase_otros` y bloques | proporción de los **otros** SKUs de la misma clase en promoción |
| $S^{\text{cat}}_{it}$ | `promo_cat_otras_clases` y bloques | proporción de los SKUs de las **otras clases** de la categoría en promoción |

**Estimador:** PPML con efectos fijos, una ecuación por categoría (D‑08), errores agrupados por SKU y por semana (D‑15). Un coeficiente $\beta$ se lee como un cambio de $100\,(e^{\beta}-1)\,\%$ en la venta esperada cuando la variable sube en una unidad.

> **Advertencia (D‑A, pendiente del profesor guía).** En las cuatro categorías con programa, los bloques de rezago se construyeron incluyendo el día de programa, así que sus **rezagos y su efecto de 7 días no se interpretan**. La evidencia está en el Anexo.

Las seis categorías sin programa ya se estimaron en `05_fase1`; aquí se vuelven a estimar con la misma especificación para tener las 10 en una sola corrida (y la matriz de varianzas que necesita el efecto de 7 días). El Anexo verifica que coinciden. Las salidas llevan el prefijo `fase1_10cat_`.

## Paso 1. Librerías y parámetros

Todos los elementos del modelo (categorías, regresores, controles, efectos fijos y agrupamiento) se definen aquí, en un solo lugar. Son los de `05_fase1`; solo cambia la lista de categorías.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import pyfixest as pf
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import stats

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# D-28, D-30: las 10 categorías objetivo del núcleo
categorias_sin_programa = ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER"]
categorias_con_programa = ["MEATS", "POULTRY", "DELI", "PRODUCE"]   # D-26: programa semanal fijo
categorias = categorias_sin_programa + categorias_con_programa
grupo = {c: "sin programa" for c in categorias_sin_programa} | {c: "con programa" for c in categorias_con_programa}

# D-14: promoción propia, de la clase y del resto de la categoría, con sus bloques de rezagos (sin rezago 7)
regresores_promocion = [
    "onpromotion", "onpromotion_l1_3", "onpromotion_l4_6",
    "promo_clase_otros", "promo_clase_otros_l1_3", "promo_clase_otros_l4_6",
    "promo_cat_otras_clases", "promo_cat_otras_clases_l1_3", "promo_cat_otras_clases_l4_6",
]

# D-10 y D-13: controles de calendario (sin transactions, D-12, ni petróleo, D-11)
controles_calendario = [
    "dia_libre", "adicional", "dia_laboral_recuperado", "evento_comercial", "evento_otro", "dia_pago", "post_pago",
]

variable_dependiente = "unit_sales_ppml"   # D-08, D-20
efectos_fijos = "item_nbr + dow + mes"     # D-16
agrupamiento = "item_nbr + semana"         # D-15

# Rutas (relativas a version_limpia); prefijo propio para no sobrescribir las salidas de 05_fase1
ruta_procesados = os.path.join("..", "data", "procesados")
ruta_tablas = os.path.join("resultados", "tablas")
ruta_figuras = os.path.join("resultados", "figuras")
prefijo = "fase1_10cat_"
os.makedirs(ruta_tablas, exist_ok=True)
os.makedirs(ruta_figuras, exist_ok=True)

## Paso 2. Base de estimación

Se reutilizan las tablas de los notebooks `01` a `03` (no se modifican) y se arma la base como indica `INSTRUCCIONES.md` §6:

1. Filas del panel de las 10 categorías, unidas con las exposiciones (por SKU y fecha) y con el calendario (por fecha).
2. Se excluyen la ventana del terremoto (D‑09) y, **solo en las categorías con programa**, sus días de programa (A‑03).
3. Se excluyen las filas con alguna variable vacía: rezagos que tocan el inicio del período o el terremoto, y SKUs sin otros SKUs activos en su clase.

Los `assert` comprueban que las uniones no dupliquen ni pierdan filas.

In [2]:
panel = pd.read_parquet(os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet"))
exposiciones = pd.read_parquet(os.path.join(ruta_procesados, "exposiciones_t44_sku_dia.parquet"))
calendario = pd.read_csv(os.path.join(ruta_procesados, "calendario_t44.csv"), parse_dates=["date"])
for tabla in (panel, exposiciones, calendario):
    tabla["date"] = tabla["date"].astype("datetime64[ns]")   # mismo formato de fecha en las tres tablas

# 1. Diez categorías, con exposiciones (1 a 1) y calendario (muchos a 1)
base = panel.loc[panel["family"].isin(categorias)].copy()
base["family"] = base["family"].astype(str)
filas_categorias = len(base)
base = base.merge(exposiciones, on=["item_nbr", "date"], how="left", validate="1:1")
base = base.merge(calendario, on="date", how="left", validate="m:1")
assert len(base) == filas_categorias, "una unión cambió el número de filas"
assert not base.duplicated(["item_nbr", "date"]).any(), "hay más de una fila por SKU-día"
assert not base.loc[base["family"].isin(categorias_sin_programa), "dia_programa"].any(), "días de programa inesperados"
base_completa = base   # se guarda para el diagnóstico del Anexo

# 2-3. Excluir terremoto (D-09), días de programa (A-03) y filas con variables vacías
variables_modelo = [variable_dependiente] + regresores_promocion + controles_calendario
base = base.loc[~base["ventana_terremoto"] & ~base["dia_programa"]].dropna(subset=variables_modelo).copy()
for columna in ["onpromotion"] + controles_calendario:
    base[columna] = base[columna].astype("int8")   # indicadores como 0/1 para la fórmula

resumen_base = (
    base.groupby("family")
    .agg(skus=("item_nbr", "nunique"), sku_dias=("item_nbr", "size"), dias=("date", "nunique"),
         sku_dias_en_promocion=("onpromotion", "sum"))
    .reindex(categorias)
)
resumen_base.insert(0, "grupo", [grupo[c] for c in categorias])
resumen_base.rename_axis("categoria").reset_index().to_csv(os.path.join(ruta_tablas, prefijo + "base_estimacion.csv"),
                                                          index=False)
print(f"Filas de las 10 categorías: {filas_categorias:,} -> base de estimación: {len(base):,}\n")
print(resumen_base.to_string())
assert (resumen_base.loc[categorias_sin_programa, "dias"] == 532).all(), "las categorías sin programa deben usar 532 días"
assert (resumen_base.loc[categorias_con_programa, "dias"] < 532).all(), "faltó excluir los días de programa"

Filas de las 10 categorías: 1,591,330 -> base de estimación: 1,391,645

                         grupo  skus  sku_dias  dias  sku_dias_en_promocion
family                                                                     
BEVERAGES         sin programa   576    258915   532                  19386
GROCERY I         sin programa  1153    556533   532                  38391
CLEANING          sin programa   396    197454   532                  13711
DAIRY             sin programa   226    109779   532                  10630
BREAD/BAKERY      sin programa   105     54466   532                   2816
LIQUOR,WINE,BEER  sin programa    49     21466   532                   1697
MEATS             con programa    55     24273   455                   1184
POULTRY           con programa    48     20977   455                    265
DELI              con programa    76     33486   455                   3166
PRODUCE           con programa   261    114296   456                   2808


**Salida.** Las categorías sin programa usan **532 días** (590 menos 46 del terremoto y 12 sin rezagos completos); las con programa, 455–456, porque además pierden sus viernes o miércoles. `sku_dias_en_promocion` es la variación que identifica el efecto propio: fuera del programa es escasa, sobre todo en POULTRY (265 SKU‑días), lo que anticipa estimaciones imprecisas.

## Paso 3. Estimar el modelo

Una ecuación por categoría con `pf.fepois`. La fórmula tiene la forma `y ~ regresores | efectos fijos`: lo que va después de la barra se absorbe como efecto fijo. `vcov={"CRV1": "item_nbr + semana"}` pide errores agrupados en dos vías.

La celda verifica que cada modelo **converja** y que los 9 coeficientes de promoción tengan error estándar válido. Avisos posibles de `pyfixest`:

- `singleton fixed effect(s) dropped` / `observations removed because of separation`: descarta SKUs con una sola observación o combinaciones donde la venta es siempre 0 (1 fila en BEVERAGES, 2 en GROCERY I, 188 en MEATS).
- `invalid value encountered in sqrt`: la varianza en dos vías sale negativa para algún **control de calendario** que marca uno o dos días; no afecta a los coeficientes de promoción. Se muestra una vez y luego se silencia.

In [3]:
formula = f"{variable_dependiente} ~ {' + '.join(regresores_promocion + controles_calendario)} | {efectos_fijos}"
print(formula, "\n")

modelos, filas = {}, []
for categoria in categorias:
    datos_j = base.loc[base["family"] == categoria]
    modelo = pf.fepois(formula, data=datos_j, vcov={"CRV1": agrupamiento})
    modelos[categoria] = modelo
    ee = modelo.se()[regresores_promocion]
    filas.append(
        {
            "categoria": categoria,
            "convergio": bool(modelo._convergence),
            "obs_modelo": int(modelo._N),
            "obs_descartadas": len(datos_j) - int(modelo._N),
            "gl_t": int(modelo._df_t),   # clústeres de la dimensión más chica - 1
            "ee_promocion_validos": bool(np.isfinite(ee).all() and (ee > 0).all()),
        }
    )

convergencia = pd.DataFrame(filas)
convergencia.to_csv(os.path.join(ruta_tablas, prefijo + "convergencia.csv"), index=False)
print(convergencia.to_string(index=False))
assert convergencia["convergio"].all(), "algún modelo no convergió"
assert convergencia["ee_promocion_validos"].all(), "algún coeficiente de promoción no tiene error estándar válido"
warnings.filterwarnings("ignore", message="invalid value encountered in sqrt")

unit_sales_ppml ~ onpromotion + onpromotion_l1_3 + onpromotion_l4_6 + promo_clase_otros + promo_clase_otros_l1_3 + promo_clase_otros_l4_6 + promo_cat_otras_clases + promo_cat_otras_clases_l1_3 + promo_cat_otras_clases_l4_6 + dia_libre + adicional + dia_laboral_recuperado + evento_comercial + evento_otro + dia_pago + post_pago | item_nbr + dow + mes 



/usr/local/lib/python3.13/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
/usr/local/lib/python3.13/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 188 observations removed because of separation.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


       categoria  convergio  obs_modelo  obs_descartadas  gl_t  ee_promocion_validos
       BEVERAGES       True      258914                1    77                  True
       GROCERY I       True      556531                2    77                  True
        CLEANING       True      197454                0    77                  True
           DAIRY       True      109779                0    77                  True
    BREAD/BAKERY       True       54466                0    77                  True
LIQUOR,WINE,BEER       True       21466                0    48                  True
           MEATS       True       24085              188    53                  True
         POULTRY       True       20977                0    47                  True
            DELI       True       33486                0    75                  True
         PRODUCE       True      114296                0    77                  True


**Salida.** Los diez modelos convergen. `gl_t` son los grados de libertad de la distribución $t$: 77 cuando la dimensión más chica del agrupamiento son las 78 semanas, y menos en LIQUOR,WINE,BEER, MEATS, POULTRY y DELI, que tienen menos SKUs que semanas.

## Paso 4. Coeficientes del modelo

Para cada categoría y cada variable de promoción (`fase1_10cat_coeficientes.csv`):

| Columna | Qué es |
|---|---|
| `beta`, `ee` | coeficiente y error estándar agrupado por SKU y semana |
| `p_valor` | con la distribución $t$ de `gl_t` grados de libertad, sin corrección múltiple (D‑15 la exige solo en la Fase 2) |
| `ic_inf`, `ic_sup` | intervalo de confianza de 95 % de `beta` |
| `efecto_pct` (+ IC) | $100\,(e^{\beta}-1)$: cambio % de la venta cuando la variable sube en 1 |
| `efecto_10pp_pct` (+ IC) | solo para $S$: $100\,(e^{0{,}1\beta}-1)$, cambio % si 10 puntos porcentuales más de los otros SKUs están en promoción |
| `interpretable` | `no (D-A)` en los rezagos de las categorías con programa |

In [4]:
filas = []
for categoria, modelo in modelos.items():
    beta = modelo.coef()[regresores_promocion]
    ee = modelo.se()[regresores_promocion]
    t_critico = stats.t.ppf(0.975, modelo._df_t)
    filas.append(
        pd.DataFrame(
            {
                "categoria": categoria,
                "grupo": grupo[categoria],
                "coeficiente": regresores_promocion,
                "beta": beta.values,
                "ee": ee.values,
                "p_valor": 2 * stats.t.sf(np.abs(beta / ee).values, modelo._df_t),
                "ic_inf": (beta - t_critico * ee).values,
                "ic_sup": (beta + t_critico * ee).values,
                "n_obs": modelo._N,
            }
        )
    )
coeficientes = pd.concat(filas, ignore_index=True)

# Efectos en %: por unidad y, para las proporciones S, por +10 pp
es_proporcion = coeficientes["coeficiente"].str.startswith("promo_")
for sufijo, escala in [("", 1.0), ("_10pp", 0.1)]:
    for columna, origen in [("", "beta"), ("_ic_inf", "ic_inf"), ("_ic_sup", "ic_sup")]:
        valores = 100 * np.expm1(escala * coeficientes[origen])
        coeficientes[f"efecto{sufijo}_pct{columna}"] = valores if sufijo == "" else valores.where(es_proporcion)

# D-A: los rezagos de las categorías con programa no se interpretan
es_rezago = coeficientes["coeficiente"].str.contains("_l")
coeficientes["interpretable"] = np.where((coeficientes["grupo"] == "con programa") & es_rezago, "no (D-A)", "sí")
coeficientes.round(6).to_csv(os.path.join(ruta_tablas, prefijo + "coeficientes.csv"), index=False)

with pd.option_context("display.float_format", lambda x: f"{x:,.3f}"):
    print(coeficientes.set_index(["categoria", "coeficiente"])[
        ["beta", "ee", "p_valor", "ic_inf", "ic_sup", "efecto_pct", "efecto_10pp_pct", "interpretable"]
    ].to_string())

                                               beta    ee  p_valor  ic_inf  ic_sup  efecto_pct  efecto_10pp_pct interpretable
categoria        coeficiente                                                                                                 
BEVERAGES        onpromotion                  0.469 0.058    0.000   0.353   0.584      59.816              NaN            sí
                 onpromotion_l1_3             0.238 0.062    0.000   0.115   0.361      26.923              NaN            sí
                 onpromotion_l4_6            -0.105 0.063    0.100  -0.229   0.020      -9.923              NaN            sí
                 promo_clase_otros           -0.306 0.114    0.009  -0.533  -0.078     -26.327           -3.009            sí
                 promo_clase_otros_l1_3      -0.020 0.120    0.871  -0.259   0.220      -1.949           -0.197            sí
                 promo_clase_otros_l4_6       0.079 0.101    0.441  -0.123   0.281       8.175            0.789       

**Cómo leer la salida.** En BEVERAGES `onpromotion` tiene $\beta = 0{,}469$: el día de la promoción la venta esperada del SKU es $e^{0{,}469} \approx 1{,}60$ veces la de un día comparable sin promoción (+60 %), con $p<0{,}001$.

## Paso 5. Resumen por tipo de promoción

Tres tablas, una por tipo de promoción, con el **día de la promoción** y los **rezagos 1–3 y 4–6** (efecto, IC 95 % y valor p):

- **Tabla 1. Efecto propio** (`onpromotion`), más el **efecto de 7 días sostenidos**, $100\,(e^{\beta_0+\beta_{\ell_{1\text{-}3}}+\beta_{\ell_{4\text{-}6}}}-1)$: la venta de un día en que el SKU lleva 7 días seguidos en promoción (`metodologia.tex` lo llama "efecto neto"; no es el efecto de una promoción de un solo día). Su IC usa el método delta; si la varianza de la suma sale negativa, se informa "n.d.".
- **Tabla 2. Otros SKU de la misma clase** (`promo_clase_otros`), por +10 pp.
- **Tabla 3. Otras clases de la categoría** (`promo_cat_otras_clases`), por +10 pp.

In [5]:
def efecto_sostenido(modelo) -> tuple:
    """
    Calcula el efecto en % de una promoción sostenida 7 días, su IC de 95 % y su valor p (método delta).

    :param modelo:  un modelo estimado con pyfixest.
    :return:        (efecto %, límite inferior %, límite superior %, valor p); NaN si la varianza de la suma no es positiva.
    """

    variables = ["onpromotion", "onpromotion_l1_3", "onpromotion_l4_6"]
    indices = [modelo._coefnames.index(v) for v in variables]
    suma = modelo.coef()[variables].sum()
    varianza = modelo._vcov[np.ix_(indices, indices)].sum()
    if varianza <= 0:
        return 100 * np.expm1(suma), np.nan, np.nan, np.nan
    ee_suma = np.sqrt(varianza)
    t_critico = stats.t.ppf(0.975, modelo._df_t)
    return (100 * np.expm1(suma), 100 * np.expm1(suma - t_critico * ee_suma),
            100 * np.expm1(suma + t_critico * ee_suma), 2 * stats.t.sf(abs(suma / ee_suma), modelo._df_t))


def formato_pct(x: float) -> str:
    """Escribe un efecto en % con signo y dos decimales."""
    return f"{x:+.2f} %"


def formato_ic(inferior: float, superior: float) -> str:
    """Escribe un intervalo en %, o 'n.d.' si no está definido."""
    return "n.d." if np.isnan(inferior) else f"[{inferior:.2f}, {superior:.2f}]"


def formato_p(p: float) -> str:
    """Escribe un valor p con tres decimales, o <0.001."""
    return "n.d." if np.isnan(p) else ("<0.001" if p < 0.001 else f"{p:.3f}")


def tabla_por_tipo(variable: str, por_10pp: bool) -> pd.DataFrame:
    """
    Arma la tabla de un tipo de promoción: día, rezago 1-3 y rezago 4-6, cada uno con efecto, IC 95 % y valor p.

    :param variable:    el nombre base de la variable (p. ej., 'onpromotion').
    :param por_10pp:    True para expresar el efecto por +10 pp (variables S); False para hacerlo por unidad.
    :return:            un DataFrame con una fila por categoría.
    """

    c = coeficientes.set_index(["categoria", "coeficiente"])
    columna = "efecto_10pp_pct" if por_10pp else "efecto_pct"
    filas = []
    for categoria in categorias:
        fila = {"Categoría": categoria, "Grupo": grupo[categoria]}
        for sufijo, nombre in [("", "Día de la promoción"), ("_l1_3", "Rezago 1-3"), ("_l4_6", "Rezago 4-6")]:
            r = c.loc[(categoria, variable + sufijo)]
            fila[nombre] = formato_pct(r[columna])
            fila[f"IC 95 % ({nombre})"] = formato_ic(r[f"{columna}_ic_inf"], r[f"{columna}_ic_sup"])
            fila[f"valor p ({nombre})"] = formato_p(r["p_valor"])
        filas.append(fila)
    return pd.DataFrame(filas)


# Efecto de 7 días sostenidos por categoría
sostenido = pd.DataFrame(
    [(c, *efecto_sostenido(modelos[c])) for c in categorias],
    columns=["categoria", "efecto_pct", "ic_inf_pct", "ic_sup_pct", "p_valor"],
).set_index("categoria")
sostenido.round(6).reset_index().to_csv(os.path.join(ruta_tablas, prefijo + "efecto_7dias_sostenidos.csv"), index=False)

tabla_1 = tabla_por_tipo("onpromotion", por_10pp=False)
tabla_1["Efecto 7 días sostenidos"] = [formato_pct(sostenido.loc[c, "efecto_pct"]) for c in categorias]
tabla_1["IC 95 % (7 días)"] = [formato_ic(sostenido.loc[c, "ic_inf_pct"], sostenido.loc[c, "ic_sup_pct"]) for c in categorias]
tabla_1["valor p (7 días)"] = [formato_p(sostenido.loc[c, "p_valor"]) for c in categorias]
tabla_2 = tabla_por_tipo("promo_clase_otros", por_10pp=True)
tabla_3 = tabla_por_tipo("promo_cat_otras_clases", por_10pp=True)

tablas = [
    ("Tabla 1. Efecto propio de la promoción (variación % de la venta esperada del SKU)", tabla_1, "tabla1_efecto_propio"),
    ("Tabla 2. Otros SKU de la misma clase (variación % por +10 pp de la clase en promoción)", tabla_2, "tabla2_misma_clase"),
    ("Tabla 3. Otras clases de la categoría (variación % por +10 pp en promoción)", tabla_3, "tabla3_otras_clases"),
]
for titulo, tabla, nombre in tablas:
    # En pantalla se omiten los IC para que la tabla quepa; están en el CSV y en las imágenes
    print(titulo)
    vista = tabla.loc[:, ~tabla.columns.str.startswith("IC 95")]
    print(vista.rename(columns=lambda x: "valor p" if x.startswith("valor p") else x).to_string(index=False), "\n")
    tabla.to_csv(os.path.join(ruta_tablas, prefijo + nombre + ".csv"), index=False)

Tabla 1. Efecto propio de la promoción (variación % de la venta esperada del SKU)
       Categoría        Grupo Día de la promoción valor p Rezago 1-3 valor p Rezago 4-6 valor p Efecto 7 días sostenidos valor p
       BEVERAGES sin programa            +59.82 %  <0.001   +26.92 %  <0.001    -9.92 %   0.100                 +82.72 %  <0.001
       GROCERY I sin programa            +60.43 %  <0.001   +29.65 %  <0.001    -1.08 %   0.801                +105.75 %  <0.001
        CLEANING sin programa            +88.94 %  <0.001   +26.25 %   0.026    -2.40 %   0.727                +132.81 %  <0.001
           DAIRY sin programa            +41.22 %  <0.001    +9.13 %  <0.001    +3.10 %   0.077                 +58.88 %  <0.001
    BREAD/BAKERY sin programa            +45.45 %  <0.001   +15.55 %   0.032   +10.35 %   0.192                 +85.47 %  <0.001
LIQUOR,WINE,BEER sin programa            +75.02 %  <0.001   +14.46 %   0.070    -0.80 %   0.938                 +98.72 %  <0.001
           MEAT

## Paso 6. Gráfico y tablas como imágenes (PNG)

Guarda en `resultados/figuras/`, a 300 dpi y con el formato de `05_fase1`:

1. `fase1_10cat_grafico_efectos.png`: efecto propio (A), misma clase (B) y otras clases (C), con IC 95 %; punto lleno si $p<0{,}05$. Las categorías con programa van en la franja sombreada.
2. `fase1_10cat_tabla1_efecto_propio.png`, `..._tabla2_misma_clase.png` y `..._tabla3_otras_clases.png`: colores según el valor p (azul oscuro $p<0{,}01$, medio $p<0{,}05$, claro $p<0{,}10$, blanco no significativo) y **gris en cursiva** para lo que no se interpreta (D‑A).

La primera celda solo define colores y funciones de dibujo; la segunda guarda las imágenes.

In [6]:
# Colores, formatos y funciones de dibujo (sin salida)
DECIMALES = 1   # decimales de los % en las imágenes

AZUL, ROJO, VERDE = "#1f5fa6", "#c0392b", "#27ae60"
AZUL_OSC, AZUL_MED, AZUL_CLA = "#2b6a9e", "#8fb5db", "#dbe8f5"
NAVY, ORO_OSC, ORO_CLA = "#13213c", "#8a6914", "#f6d98a"
FONDO_CAT, BORDE, GRIS, FRANJA = "#eef3f9", "#c5cfdb", "#e3e3e3", "#f3efe4"
nombres_vista = {
    "BEVERAGES": "Beverages", "GROCERY I": "Grocery I", "CLEANING": "Cleaning", "DAIRY": "Dairy",
    "BREAD/BAKERY": "Bread/Bakery", "LIQUOR,WINE,BEER": "Liquor, Wine, Beer",
    "MEATS": "Meats*", "POULTRY": "Poultry*", "DELI": "Deli*", "PRODUCE": "Produce*",
}
ANCHO_TAB, ALTO_CAB, ANCHO_FIG, MARGEN = 10.9, 0.75, 11.5, 0.3   # en pulgadas


def pct(x: float) -> str:
    """Escribe un efecto en % con signo (menos tipográfico) y DECIMALES decimales."""
    return f"{x:+.{DECIMALES}f} %".replace("-", "−")


def estilo_p(p: float, interpretable: bool = True):
    """Devuelve (fondo, color de texto, peso, estilo) según el valor p mostrado y si el coeficiente se interpreta."""
    if not interpretable:
        return GRIS, "#555555", "normal", "italic"
    p = round(p, 3)
    if p < 0.01:
        return AZUL_OSC, "white", "bold", "normal"
    if p < 0.05:
        return AZUL_MED, "#111111", "normal", "normal"
    if p < 0.10:
        return AZUL_CLA, "#111111", "normal", "normal"
    return "white", "#111111", "normal", "normal"


def ejes_en(fig, W, H, x0, y_top, ancho, alto, **kw):
    """Crea un eje con esquina superior izquierda en (x0, y_top) y tamaño (ancho, alto), todo en pulgadas."""
    return fig.add_axes([x0 / W, 1 - (y_top + alto) / H, ancho / W, alto / H], **kw)


def guardar(fig, nombre: str) -> None:
    """Guarda la figura como PNG (300 dpi) en resultados/figuras y la cierra, sin mostrarla."""
    ruta = os.path.join(ruta_figuras, prefijo + nombre + ".png")
    fig.savefig(ruta, dpi=300)
    plt.close(fig)
    print("Guardada:", ruta)


def grafico_efectos() -> None:
    """Dibuja y guarda el gráfico de efectos (paneles A, B y C) de las 10 categorías."""
    paneles = [
        ("A. Efecto propio (día)", "onpromotion", "efecto_pct", "Cambio en la venta esperada del SKU (%)"),
        ("B. Misma clase", "promo_clase_otros", "efecto_10pp_pct", "Cambio por +10 pp de la\nclase en promoción (%)"),
        ("C. Otras clases de la categoría", "promo_cat_otras_clases", "efecto_10pp_pct",
         "Cambio por +10 pp de otras\nclases en promoción (%)"),
    ]
    H = 7.1
    fig = plt.figure(figsize=(ANCHO_FIG, H), facecolor="white")
    fig.text(MARGEN / ANCHO_FIG, 1 - 0.35 / H,
             "Efectos asociados a la promoción por categoría (PPML, IC 95 %, EE agrupados por SKU y semana)",
             fontsize=11, va="center")
    posiciones = np.arange(len(categorias))[::-1]   # la primera categoría arriba
    for k, (titulo, variable, columna, etiqueta_x) in enumerate(paneles):
        d = coeficientes[coeficientes["coeficiente"] == variable].set_index("categoria").loc[categorias]
        eje = ejes_en(fig, ANCHO_FIG, H, 1.9 + k * 3.0, 0.95, 2.55, 4.75)
        eje.axhspan(-0.75, len(categorias_con_programa) - 0.5, color=FRANJA, zorder=0)
        for posicion, categoria in zip(posiciones, categorias):
            efecto, p = d.loc[categoria, columna], d.loc[categoria, "p_valor"]
            color = AZUL if k == 0 else (VERDE if efecto > 0 else ROJO)
            eje.hlines(posicion, d.loc[categoria, f"{columna}_ic_inf"], d.loc[categoria, f"{columna}_ic_sup"],
                       color=color, linewidth=2)
            eje.plot(efecto, posicion, "o", markersize=7, color=color, markerfacecolor=color if p < 0.05 else "white",
                     markeredgewidth=1.8, zorder=3)
            eje.annotate(pct(efecto), (efecto, posicion), textcoords="offset points", xytext=(0, -12), ha="center",
                         fontsize=7, color="#444444")
        eje.axvline(0, color="#555555", linewidth=1)
        eje.set_title(titulo, loc="left", fontsize=10.5, fontweight="bold")
        eje.set_xlabel(etiqueta_x, fontsize=8.5)
        eje.set_ylim(-0.75, len(categorias) - 0.4)
        eje.set_yticks(posiciones)
        eje.set_yticklabels([nombres_vista[c] for c in categorias], fontsize=9)
        eje.tick_params(axis="y", labelleft=(k == 0), length=0)
        eje.grid(axis="x", color="#e5e5e5")
        eje.set_axisbelow(True)
        for lado in ["top", "right", "left"]:
            eje.spines[lado].set_visible(False)
    notas = [
        "Punto lleno: p < 0,05. Punto vacío: no significativo al 5 %. Efecto = 100(e^β − 1); B y C por +10 pp. "
        "Tienda 44, 2016–2017.",
        "* Franja sombreada: categorías con programa semanal fijo; solo promociones fuera del programa (455–456 días). "
        "Sin programa: 532 días.",
        "  En esas categorías, el efecto del día puede estar afectado por los rezagos que incluyen el día de programa "
        "(D‑A, pendiente).",
    ]
    for k, nota in enumerate(notas):
        fig.text(MARGEN / ANCHO_FIG, 1 - (6.5 + 0.22 * k) / H, nota, fontsize=8, color="#888888", va="center")
    guardar(fig, "grafico_efectos")


def tabla_imagen(titulo: str, variable: str, columna: str, con_sostenido: bool, alto_fila: float, nombre: str) -> None:
    """Dibuja una tabla coloreada (día, rezago 1-3, rezago 4-6 y, si corresponde, 7 días) y la guarda como PNG."""
    c = coeficientes.set_index(["categoria", "coeficiente"])
    alto_tabla = ALTO_CAB + alto_fila * len(categorias)
    H = 0.6 + alto_tabla + 1.15
    fig = plt.figure(figsize=(ANCHO_FIG, H), facecolor="white")
    fig.text(MARGEN / ANCHO_FIG, 1 - 0.35 / H, titulo, fontsize=11.5, fontweight="bold", va="center")

    # Cuadrícula de la tabla
    eje = ejes_en(fig, ANCHO_FIG, H, MARGEN, 0.6, ANCHO_TAB, alto_tabla)
    anchos = np.array([2.0] + [1.55, 1.1] * 3 + ([2.0] if con_sostenido else []))
    anchos = anchos * ANCHO_TAB / anchos.sum()
    x = np.concatenate([[0], np.cumsum(anchos)])
    eje.set_xlim(0, ANCHO_TAB)
    eje.set_ylim(alto_tabla, 0)
    eje.axis("off")

    def celda(j, y0, texto, fondo, color="#111111", peso="normal", estilo="normal", tamano=9.5):
        eje.add_patch(Rectangle((x[j], y0), anchos[j], alto_fila if y0 else ALTO_CAB, facecolor=fondo,
                                edgecolor="white" if y0 == 0 else BORDE, linewidth=0.8))
        eje.text(x[j] + anchos[j] / 2, y0 + (alto_fila if y0 else ALTO_CAB) / 2, texto, color=color, fontweight=peso,
                 fontstyle=estilo, ha="center", va="center", fontsize=tamano)

    cabeceras = ["Categoría", "Día de la\npromoción", "valor p", "Rezago 1–3", "valor p", "Rezago 4–6", "valor p"]
    for j, texto in enumerate(cabeceras + (["Efecto 7 días\nsostenidos"] if con_sostenido else [])):
        celda(j, 0, texto, ORO_OSC if j == 7 else NAVY, "white", "bold")
    for i, cat in enumerate(categorias):
        y0 = ALTO_CAB + i * alto_fila
        celda(0, y0, nombres_vista[cat], FONDO_CAT if grupo[cat] == "sin programa" else FRANJA, peso="bold")
        for b, sufijo in enumerate(["", "_l1_3", "_l4_6"]):
            r = c.loc[(cat, variable + sufijo)]
            fondo, color, peso, estilo = estilo_p(r["p_valor"], r["interpretable"] == "sí")
            celda(1 + 2 * b, y0, pct(r[columna]), fondo, color, peso, estilo)
            celda(2 + 2 * b, y0, formato_p(r["p_valor"]), fondo, color, peso, estilo)
        if con_sostenido:
            s, ok = sostenido.loc[cat], grupo[cat] == "sin programa"
            ic = "IC n.d." if np.isnan(s["ic_inf_pct"]) else f"[{s['ic_inf_pct']:.{DECIMALES}f}, {s['ic_sup_pct']:.{DECIMALES}f}]"
            celda(7, y0, f"{pct(s['efecto_pct'])}\n{ic}".replace("-", "−"), ORO_CLA if ok else GRIS,
                  "#111111" if ok else "#555555", "bold" if ok else "normal", "normal" if ok else "italic", 9)

    # Leyenda y notas al pie
    y = 0.6 + alto_tabla
    ley = ejes_en(fig, ANCHO_FIG, H, MARGEN, y + 0.1, ANCHO_TAB, 0.3)
    ley.set_xlim(0, ANCHO_TAB)
    ley.set_ylim(0.3, 0)
    ley.axis("off")
    items = [(AZUL_OSC, "p < 0.01", 0.0), (AZUL_MED, "p < 0.05", 1.6), (AZUL_CLA, "p < 0.10", 3.2),
             ("white", "no significativo", 4.8), (GRIS, "no se interpreta (D‑A)", 6.9)]
    for fondo, texto, xi in items + ([(ORO_CLA, "7 días (IC 95 %)", 9.3)] if con_sostenido else []):
        ley.add_patch(Rectangle((xi, 0.05), 0.35, 0.2, facecolor=fondo, edgecolor=BORDE, linewidth=0.8))
        ley.text(xi + 0.45, 0.15, texto, va="center", fontsize=8.5)
    notas = [
        "Efecto 7 días = e^(β₀+β₁₋₃+β₄₋₆) − 1: SKU en promoción 7 días seguidos; IC por método delta (n.d.: varianza "
        "negativa). Valor p con EE agrupados por SKU y semana, sin corrección múltiple." if con_sostenido else
        "Valor p con EE agrupados por SKU y semana, sin corrección múltiple. Efecto = 100(e^(0,1β) − 1).",
        "* Categorías con programa semanal fijo: solo promociones fuera del programa; sus rezagos incluyen el día "
        "de programa (D‑A) y no se interpretan.",
    ]
    for k, nota in enumerate(notas):
        fig.text(MARGEN / ANCHO_FIG, 1 - (y + 0.6 + 0.22 * k) / H, nota, fontsize=7.3, color="#555555", va="center")
    guardar(fig, nombre)

In [7]:
grafico_efectos()
tabla_imagen("Tabla 1. Efecto propio de la promoción (variación % de la venta esperada del SKU)",
             "onpromotion", "efecto_pct", True, 0.6, "tabla1_efecto_propio")
tabla_imagen("Tabla 2. Otros SKU de la misma clase (variación % por +10 pp de la clase en promoción)",
             "promo_clase_otros", "efecto_10pp_pct", False, 0.45, "tabla2_misma_clase")
tabla_imagen("Tabla 3. Otras clases de la categoría (variación % por +10 pp en promoción)",
             "promo_cat_otras_clases", "efecto_10pp_pct", False, 0.45, "tabla3_otras_clases")

Guardada: resultados/figuras/fase1_10cat_grafico_efectos.png


Guardada: resultados/figuras/fase1_10cat_tabla1_efecto_propio.png
Guardada: resultados/figuras/fase1_10cat_tabla2_misma_clase.png


Guardada: resultados/figuras/fase1_10cat_tabla3_otras_clases.png


## Paso 7. Interpretación de los resultados

La celda arma, sin escribir cifras a mano, una lectura de cada efecto: **signo** (aumento o disminución), **magnitud** (efecto en % con su IC) y **significancia** (1 %, 5 %, 10 % o no significativo). Se guarda en `fase1_10cat_interpretacion.csv`. Los rezagos de las categorías con programa no se leen (D‑A).

In [8]:
def lectura(fila: pd.Series, por_10pp: bool) -> str:
    """Escribe en una frase el signo, la magnitud con su IC y la significancia de un coeficiente."""
    columna = "efecto_10pp_pct" if por_10pp else "efecto_pct"
    efecto, p = fila[columna], fila["p_valor"]
    nivel = ("significativo al 1 %" if p < 0.01 else "significativo al 5 %" if p < 0.05
             else "significativo solo al 10 %" if p < 0.10 else "no significativo")
    signo = "aumento" if efecto > 0 else "disminución"
    return f"{signo} de {abs(efecto):.1f} % [IC {fila[f'{columna}_ic_inf']:.1f}; {fila[f'{columna}_ic_sup']:.1f}], {nivel}"


c = coeficientes.set_index(["categoria", "coeficiente"])
filas = []
for cat in categorias:
    rezagos = grupo[cat] == "sin programa"
    filas.append(
        {
            "categoria": cat,
            "grupo": grupo[cat],
            "propio_dia": lectura(c.loc[(cat, "onpromotion")], False),
            "propio_rezago_1_3": lectura(c.loc[(cat, "onpromotion_l1_3")], False) if rezagos else "no se interpreta (D-A)",
            "propio_rezago_4_6": lectura(c.loc[(cat, "onpromotion_l4_6")], False) if rezagos else "no se interpreta (D-A)",
            "misma_clase_dia_10pp": lectura(c.loc[(cat, "promo_clase_otros")], True),
            "otras_clases_dia_10pp": lectura(c.loc[(cat, "promo_cat_otras_clases")], True),
        }
    )
interpretacion = pd.DataFrame(filas)
interpretacion.to_csv(os.path.join(ruta_tablas, prefijo + "interpretacion.csv"), index=False)
for _, f in interpretacion.iterrows():
    print(f"{f['categoria']} ({f['grupo']})")
    for clave, nombre in [("propio_dia", "propio, día"), ("propio_rezago_1_3", "propio, rezago 1-3"),
                          ("propio_rezago_4_6", "propio, rezago 4-6"), ("misma_clase_dia_10pp", "misma clase (+10 pp)"),
                          ("otras_clases_dia_10pp", "otras clases (+10 pp)")]:
        print(f"  {nombre:.<24s} {f[clave]}")

BEVERAGES (sin programa)
  propio, día............. aumento de 59.8 % [IC 42.4; 79.4], significativo al 1 %
  propio, rezago 1-3...... aumento de 26.9 % [IC 12.2; 43.5], significativo al 1 %
  propio, rezago 4-6...... disminución de 9.9 % [IC -20.5; 2.1], significativo solo al 10 %
  misma clase (+10 pp).... disminución de 3.0 % [IC -5.2; -0.8], significativo al 1 %
  otras clases (+10 pp)... disminución de 6.1 % [IC -12.7; 0.9], significativo solo al 10 %
GROCERY I (sin programa)
  propio, día............. aumento de 60.4 % [IC 45.0; 77.5], significativo al 1 %
  propio, rezago 1-3...... aumento de 29.7 % [IC 19.1; 41.2], significativo al 1 %
  propio, rezago 4-6...... disminución de 1.1 % [IC -9.2; 7.8], no significativo
  misma clase (+10 pp).... disminución de 2.2 % [IC -3.6; -0.8], significativo al 1 %
  otras clases (+10 pp)... disminución de 7.1 % [IC -14.1; 0.5], significativo solo al 10 %
CLEANING (sin programa)
  propio, día............. aumento de 88.9 % [IC 63.0; 119.0], si

### Lectura por categoría

Las cifras son las de las Tablas 1 a 3 y de la celda anterior. Todas son **efectos asociados**: cambios de la venta esperada condicionales al SKU, al día de la semana, al mes y al calendario. "Significativo" se refiere al valor p sin corrección múltiple.

**Categorías sin programa (BEVERAGES, GROCERY I, CLEANING, DAIRY, BREAD/BAKERY, LIQUOR,WINE,BEER).** Sus resultados son idénticos a los de `05_fase1` (Anexo).

- **Efecto propio (pregunta i).** Positivo y significativo al 1 % en las seis: de **+41,2 %** (DAIRY) a **+88,9 %** (CLEANING), con BEVERAGES (+59,8 %), GROCERY I (+60,4 %), BREAD/BAKERY (+45,5 %) y LIQUOR,WINE,BEER (+75,0 %) en el medio. Los intervalos se superponen, así que el orden entre categorías es solo indicativo.
- **¿Caen las ventas después?** No se observa caída: el rezago 1–3 es positivo en las seis (al 1 % en BEVERAGES, GROCERY I y DAIRY; al 5 % en CLEANING y BREAD/BAKERY; solo al 10 % en LIQUOR,WINE,BEER) y el rezago 4–6 no es significativo al 5 % en ninguna (BEVERAGES −9,9 %, p = 0,100). Esto **no prueba** ausencia de desplazamiento intertemporal: las campañas duran varios días, y el rezago 1–3 positivo refleja sobre todo que la venta sigue alta mientras la campaña continúa.
- **7 días sostenidos.** Entre +58,9 % (DAIRY) y +132,8 % (CLEANING), todos con intervalo que excluye el 0. Es el efecto de una campaña de una semana, no el de una promoción de un día.
- **Misma clase (pregunta ii).** Negativo y significativo al 5 % en BEVERAGES (−3,0 % por +10 pp), GROCERY I (−2,2 %), CLEANING (−3,1 %) y DAIRY (−1,0 %): consistente con **canibalización entre sustitutos cercanos**, de magnitud pequeña. En BREAD/BAKERY no es significativo. En LIQUOR,WINE,BEER es **positivo** (+7,9 %, p = 0,001), pero es un resultado frágil: muchas de sus clases tienen un solo SKU y esas filas salen del modelo (solo entran 49 de sus 60 SKUs), así que el coeficiente se identifica con pocos SKUs; no debe leerse como sinergia (D‑E).
- **Otras clases de la categoría (pregunta ii).** Negativo en las seis, pero solo significativo al 5 % en BREAD/BAKERY (−6,4 % por +10 pp, p = 0,005); en BEVERAGES y GROCERY I lo es solo al 10 %. Con estos datos no se puede afirmar un efecto sistemático sobre el resto de la categoría.

**Categorías con programa (MEATS, POULTRY, DELI, PRODUCE): solo promociones fuera del programa.** Dos advertencias se aplican a las cuatro: los rezagos y el efecto de 7 días no se interpretan (D‑A, Anexo), y el efecto del día puede estar afectado por esos mismos rezagos, porque los coeficientes se estiman en conjunto.

- **MEATS.** El efecto propio fuera del viernes es **+6,9 %, no significativo** (p = 0,398; IC −8,6 % a +25,1 %), coherente con la razón descriptiva cercana a 1 (1,12). Los efectos de la clase (−0,4 %) y de otras clases (+1,4 %) tampoco son significativos. Con los datos disponibles, las promociones de carnes fuera del viernes no se asocian a un aumento detectable de la venta.
- **POULTRY.** **−17,3 %, no significativo** (p = 0,108). Es la categoría con menos variación: solo 265 SKU‑días en promoción fuera del viernes (1,3 % de la muestra, en 20 SKUs), y su matriz de varianzas no es definida positiva (Anexo). Es el único caso en que el signo no coincide con el descriptivo (Anexo), pero el intervalo es amplio y no permite concluir nada sobre el efecto propio.
- **DELI.** **+18,9 %, significativo al 1 %** (IC +6,7 % a +32,6 %). Es la categoría con programa con más promociones fuera del viernes (3.166 SKU‑días) y su resultado es el más creíble de las cuatro: un efecto positivo, pero menor que en las categorías sin programa. La clase (+0,2 %) no es significativa; otras clases (−4,1 %) lo es solo al 10 %.
- **PRODUCE.** **−47,2 %, significativo al 1 %**. Es un resultado contraintuitivo y, siguiendo `INSTRUCCIONES.md` §10.1, **no debe reportarse como efecto** hasta resolver D‑A. Dos hechos lo explican en parte: (1) la razón descriptiva también es menor que 1 (0,79), es decir, los SKUs de PRODUCE que se promocionan fuera del miércoles venden menos que el resto; (2) el rezago 1–3 es enorme y de signo opuesto (+143,8 %), la señal típica de dos regresores correlacionados que se compensan, y ese rezago es justamente el que incluye el miércoles de programa. La clase sale positiva (+3,5 %, p = 0,019), también contra lo esperado; con 90 coeficientes de promoción, un valor p de 0,02 aislado puede ser un falso positivo.

**Síntesis para las preguntas (i) y (ii).** En las seis categorías sin programa la promoción se asocia a aumentos grandes de la venta del propio SKU (+41 % a +89 % el día), sin caída posterior detectable, y a una canibalización pequeña pero sistemática dentro de la clase (−1 % a −3 % por +10 pp) en cuatro de ellas. En las categorías con programa, las promociones fuera del programa tienen efectos propios menores o no detectables (DELI es la única con un aumento claro), y sus resultados dependen de la decisión pendiente D‑A.

## Anexo. Verificaciones

No forman parte del modelo; respaldan que los resultados son confiables (`INSTRUCCIONES.md` §10.1) y documentan el problema D‑A para el profesor guía. Una sola celda hace tres controles:

1. **Reproducción de `05_fase1`.** Las seis categorías sin programa deben coincidir con `fase1_coeficientes.csv` (redondeado a 6 decimales; se acepta $10^{-5}$).
2. **Comparación con los descriptivos.** La dirección del efecto propio ($e^{\beta_0}$) frente a la razón de medias con/sin promoción de la tabla 4 de `04_descriptivos` (`fase1_10cat_control_descriptivos.csv`).
3. **Diagnóstico D‑A.** En las categorías con programa, media del bloque de rezago propio en las filas cuyo bloque **incluye** un día de programa frente a las que no (`fase1_10cat_diagnostico_rezagos_programa.csv` y `.png`).

In [9]:
# 1. Reproducción de 05_fase1
coeficientes_05 = pd.read_csv(os.path.join(ruta_tablas, "fase1_coeficientes.csv")).set_index(["categoria", "coeficiente"])
nuevo = coeficientes.set_index(["categoria", "coeficiente"]).loc[coeficientes_05.index]
dif = (nuevo[["beta", "ee"]] - coeficientes_05[["beta", "ee"]]).abs().max()
print(f"1. Máxima diferencia con 05_fase1: beta {dif['beta']:.1e}, ee {dif['ee']:.1e}")
assert (dif < 1e-5).all(), "la estimación no reproduce 05_fase1"

# 2. Comparación con los descriptivos (tabla 4 de 04_descriptivos)
control = (
    pd.read_csv(os.path.join(ruta_tablas, "tabla_04_ventas_con_sin_promocion.csv"))
    .set_index("family").reindex(categorias)[["razon_medias"]].rename(columns={"razon_medias": "razon_descriptiva"})
)
control["razon_modelo"] = [np.exp(modelos[cat].coef()["onpromotion"]) for cat in categorias]
control["misma_direccion"] = np.sign(control["razon_descriptiva"] - 1) == np.sign(control["razon_modelo"] - 1)
control.round(6).rename_axis("categoria").reset_index().to_csv(
    os.path.join(ruta_tablas, prefijo + "control_descriptivos.csv"), index=False
)
print("\n2. Efecto propio: razón descriptiva vs. modelo")
print(control.round(3).to_string())

# 3. Diagnóstico D-A: ¿el bloque de rezago incluye un día de programa de la categoría?
indice_dia = pd.Series(np.arange(len(calendario)), index=calendario.sort_values("date")["date"])
filas = []
for cat in categorias_con_programa:
    t_programa = set(indice_dia[base_completa.loc[(base_completa["family"] == cat) & base_completa["dia_programa"], "date"].unique()])
    datos_c = base.loc[base["family"] == cat]
    t_fila = indice_dia[datos_c["date"]].to_numpy()
    for sufijo, desplazamientos in [("l1_3", range(1, 4)), ("l4_6", range(4, 7))]:
        incluye = np.array([any((t - k) in t_programa for k in desplazamientos) for t in t_fila])
        for valor, etiqueta in [(False, "no"), (True, "sí")]:
            filas.append({"categoria": cat, "bloque": sufijo, "incluye_dia_programa": etiqueta,
                          "sku_dias": int((incluye == valor).sum()),
                          "media_onpromotion_bloque": datos_c.loc[incluye == valor, f"onpromotion_{sufijo}"].mean(),
                          "media_onpromotion_dia": datos_c.loc[incluye == valor, "onpromotion"].mean()})
diagnostico = pd.DataFrame(filas)
diagnostico.round(6).to_csv(os.path.join(ruta_tablas, prefijo + "diagnostico_rezagos_programa.csv"), index=False)
print("\n3. Diagnóstico D-A (medias en la muestra de estimación)")
print(diagnostico.round(3).to_string(index=False))

# Figura del diagnóstico
fig, ejes = plt.subplots(1, 2, figsize=(ANCHO_FIG, 3.8), facecolor="white", sharey=True)
for eje, (sufijo, nombre) in zip(ejes, [("l1_3", "Bloque de rezago 1–3"), ("l4_6", "Bloque de rezago 4–6")]):
    d = diagnostico[diagnostico["bloque"] == sufijo].set_index(["categoria", "incluye_dia_programa"])
    x = np.arange(len(categorias_con_programa))
    for desplazamiento, etiqueta, color in [(-0.19, "no", "#9bb7d4"), (0.19, "sí", ORO_OSC)]:
        barras = eje.bar(x + desplazamiento, [d.loc[(cat, etiqueta), "media_onpromotion_bloque"]
                                              for cat in categorias_con_programa], 0.38, color=color,
                         label=f"el bloque {'incluye' if etiqueta == 'sí' else 'no incluye'} un día de programa")
        eje.bar_label(barras, fmt="%.2f", fontsize=8, padding=2)
    eje.set_xticks(x)
    eje.set_xticklabels([nombres_vista[cat].rstrip("*") for cat in categorias_con_programa])
    eje.set_title(nombre, loc="left", fontsize=10.5, fontweight="bold")
    for lado in ["top", "right"]:
        eje.spines[lado].set_visible(False)
ejes[0].set_ylim(0, 0.5)
ejes[0].set_ylabel("Media del bloque de promoción propia", fontsize=9)
ejes[0].legend(fontsize=8, frameon=False, loc="upper left", ncol=2)
fig.suptitle("Los rezagos de las categorías con programa incluyen el día de programa (muestra de estimación)",
             x=MARGEN / ANCHO_FIG, ha="left", fontsize=11)
fig.text(MARGEN / ANCHO_FIG, 0.02, "Discrepancia D‑A, pendiente de decisión del profesor guía.", fontsize=8, color="#888888")
fig.tight_layout(rect=(0, 0.05, 1, 0.93))
guardar(fig, "diagnostico_rezagos_programa")

1. Máxima diferencia con 05_fase1: beta 4.9e-07, ee 5.0e-07

2. Efecto propio: razón descriptiva vs. modelo
                  razon_descriptiva  razon_modelo  misma_direccion
family                                                            
BEVERAGES                      2.35         1.598             True
GROCERY I                      2.36         1.604             True
CLEANING                       2.36         1.889             True
DAIRY                          1.28         1.412             True
BREAD/BAKERY                   1.91         1.455             True
LIQUOR,WINE,BEER               2.20         1.750             True
MEATS                          1.12         1.069             True
POULTRY                        1.11         0.827            False
DELI                           1.39         1.189             True
PRODUCE                        0.79         0.528             True



3. Diagnóstico D-A (medias en la muestra de estimación)
categoria bloque incluye_dia_programa  sku_dias  media_onpromotion_bloque  media_onpromotion_dia
    MEATS   l1_3                   no     12112                     0.051                  0.047
    MEATS   l1_3                   sí     12161                     0.354                  0.051
    MEATS   l4_6                   no     12161                     0.049                  0.051
    MEATS   l4_6                   sí     12112                     0.354                  0.047
  POULTRY   l1_3                   no     10462                     0.013                  0.013
  POULTRY   l1_3                   sí     10515                     0.323                  0.012
  POULTRY   l4_6                   no     10515                     0.014                  0.012
  POULTRY   l4_6                   sí     10462                     0.324                  0.013
     DELI   l1_3                   no     16708                     0.

Guardada: resultados/figuras/fase1_10cat_diagnostico_rezagos_programa.png


**Lectura del Anexo.**

1. Las diferencias con `05_fase1` son solo de redondeo: la especificación es la misma.
2. La dirección del efecto propio coincide con los descriptivos en nueve de diez categorías; la excepción es POULTRY, cuyo efecto no es significativo. En PRODUCE también la razón descriptiva es menor que 1 (0,79).
3. En las cuatro categorías con programa, el bloque de rezago vale entre 0,31 y 0,38 cuando incluye un día de programa y entre 0,01 y 0,09 cuando no, aunque la promoción del mismo día es igual en ambos grupos. Ese exceso viene del programa y no de promociones fuera de él: es el problema D‑A. Las opciones para el profesor guía son mantener A‑03 y no interpretar esas categorías, fijar en 0 los días de programa al construir los rezagos, o separar cada bloque en un componente de programa y otro fuera de programa.

**Otras limitaciones.** En MEATS y POULTRY la matriz de varianzas en dos vías no es definida positiva (la corrección de Cameron‑Gelbach‑Miller está propuesta y no aprobada, D‑B), lo que afecta al IC del efecto de 7 días. Son **efectos asociados**, no causales, para la tienda 44 en 2016–2017; el efecto de los programas semanales mismos no es estimable y no debe presentarse como nulo.

In [10]:
print(f"\n>>>> Proceso finalizado <<<<")


>>>> Proceso finalizado <<<<
